In [ ]:
%pip install -q monai nibabel scikit-learn pandas matplotlib tqdm

In [ ]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from monai.data import Dataset as MonaiDataset
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    ResizeWithPadOrCropd,
    RandFlipd,
    EnsureTyped,
)

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA GPU detected. 3D MRI training may be very slow on CPU.")

In [ ]:
# Update these two paths to your local dataset.
METADATA_CSV = Path("data/metadata.csv")
IMAGE_ROOT = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["CN", "MCI", "AD"]
LABEL_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

# Processing/model configuration. The 96^3 tensor is a compromise between coverage and memory.
TARGET_SPACING = (2.0, 2.0, 2.0)  # millimetres per voxel
ROI_SIZE = (96, 96, 96)
BATCH_SIZE = 1                 # conservative starting point for GPUs with limited VRAM
NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0               # robust default for Windows/Jupyter

print({"target_spacing_mm": TARGET_SPACING, "tensor_size": ROI_SIZE,
       "batch_size": BATCH_SIZE, "epochs": NUM_EPOCHS})

In [ ]:
if not METADATA_CSV.exists():
    raise FileNotFoundError(
        f"Metadata CSV not found: {METADATA_CSV.resolve()}\n"
        "Create it using the required columns and point image_path to real NIfTI files. "
        "No synthetic result metrics are generated by this notebook."
    )

df = pd.read_csv(METADATA_CSV)
required_columns = {"subject_id", "image_path", "label"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Metadata CSV is missing columns: {sorted(missing_columns)}")

if df[list(required_columns)].isna().any().any():
    raise ValueError("Metadata contains missing subject_id, image_path, or label values.")

df["subject_id"] = df["subject_id"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.upper()
unknown_labels = sorted(set(df["label"]) - set(CLASS_NAMES))
if unknown_labels:
    raise ValueError(f"Unexpected labels {unknown_labels}; expected only {CLASS_NAMES}.")


def resolve_image_path(value):
    path = Path(str(value)).expanduser()
    if not path.is_absolute():
        path = IMAGE_ROOT / path
    return path.resolve()

df["image_path"] = df["image_path"].map(resolve_image_path)
missing_files = [str(path) for path in df["image_path"] if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Could not find {len(missing_files)} MRI file(s). First few:\n" +
        "\n".join(missing_files[:5])
    )

print(f"Scans: {len(df)} | Unique participants: {df['subject_id'].nunique()}")
print("Class counts (scan-level; review if participants have repeated scans):")
print(df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0))
display(df.head())

In [ ]:
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
trainval_idx, test_idx = next(
    outer_splitter.split(df, y=df["label"], groups=df["subject_id"])
)
trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

inner_splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
train_idx, val_idx = next(
    inner_splitter.split(
        trainval_df,
        y=trainval_df["label"],
        groups=trainval_df["subject_id"],
    )
)
train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

train_subjects = set(train_df["subject_id"])
val_subjects = set(val_df["subject_id"])
test_subjects = set(test_df["subject_id"])
assert train_subjects.isdisjoint(val_subjects)
assert train_subjects.isdisjoint(test_subjects)
assert val_subjects.isdisjoint(test_subjects)

for name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"\n{name}: {len(split_df)} scans, {split_df['subject_id'].nunique()} participants")
    print(split_df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0))

# Persist split membership so the exact split can be audited/reused.
for name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    split_df[["subject_id", "image_path", "label"]].to_csv(
        OUTPUT_DIR / f"{name}_split.csv", index=False
    )

In [ ]:
common_transforms = [
    LoadImaged(keys="image"),
    EnsureChannelFirstd(keys="image", channel_dim="no_channel"),
    Orientationd(keys="image", axcodes="RAS"),
    Spacingd(keys="image", pixdim=TARGET_SPACING, mode="bilinear"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ResizeWithPadOrCropd(keys="image", spatial_size=ROI_SIZE),
    EnsureTyped(keys="image", dtype=torch.float32),
]

train_transforms = Compose(common_transforms + [
    RandFlipd(keys="image", prob=0.5, spatial_axis=0),
])
eval_transforms = Compose(common_transforms)


def to_records(frame):
    return [
        {
            "image": str(row.image_path),
            "label": LABEL_TO_ID[row.label],
            "subject_id": str(row.subject_id),
        }
        for row in frame.itertuples(index=False)
    ]

train_ds = MonaiDataset(data=to_records(train_df), transform=train_transforms)
val_ds = MonaiDataset(data=to_records(val_df), transform=eval_transforms)
test_ds = MonaiDataset(data=to_records(test_df), transform=eval_transforms)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda")
)

example_batch = next(iter(train_loader))
print("Volume batch shape:", tuple(example_batch["image"].shape))
print("Label batch shape:", tuple(example_batch["label"].shape))
assert tuple(example_batch["image"].shape[1:]) == (1, *ROI_SIZE)
assert example_batch["image"].ndim == 5  # B x C x D x H x W

In [ ]:
volume = example_batch["image"][0, 0].detach().cpu().numpy()
mid = [n // 2 for n in volume.shape]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(volume[mid[0], :, :].T, cmap="gray", origin="lower")
axes[0].set_title("Coronal-like view")
axes[1].imshow(volume[:, mid[1], :].T, cmap="gray", origin="lower")
axes[1].set_title("Sagittal-like view")
axes[2].imshow(volume[:, :, mid[2]].T, cmap="gray", origin="lower")
axes[2].set_title("Axial-like view")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(
            in_channels, out_channels, kernel_size=3,
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm3d(out_channels)
        self.act = nn.GELU()
        self.conv2 = nn.Conv3d(
            out_channels, out_channels, kernel_size=3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm3d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm3d(out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return self.act(x + residual)


class CNN3DBackbone(nn.Module):
    """For 96^3 input, downsampling by 16 produces a 6x6x6 feature grid."""
    out_channels = 128

    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm3d(16),
            nn.GELU(),
        )
        self.layer1 = ConvBlock3D(16, 32, stride=2)
        self.layer2 = ConvBlock3D(32, 64, stride=2)
        self.layer3 = ConvBlock3D(64, 128, stride=2)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x


class CNN3DClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = CNN3DBackbone()
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(self.backbone.out_channels, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(self.pool(features))

In [ ]:
class Hybrid3DCNNViT(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, transformer_layers=3, dropout=0.1):
        super().__init__()
        self.backbone = CNN3DBackbone()
        embed_dim = self.backbone.out_channels
        grid_size = tuple(size // 16 for size in ROI_SIZE)
        num_tokens = int(np.prod(grid_size))

        self.pos_embed = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=4,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(embed_dim),
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        feature_map = self.backbone(x)                  # B, C, D, H, W
        tokens = feature_map.flatten(start_dim=2)      # B, C, N
        tokens = tokens.transpose(1, 2)                # B, N, C
        if tokens.shape[1] != self.pos_embed.shape[1]:
            raise ValueError(
                f"Expected {self.pos_embed.shape[1]} tokens, got {tokens.shape[1]}. "
                "Check ROI_SIZE and backbone downsampling."
            )
        tokens = tokens + self.pos_embed
        tokens = self.transformer(tokens)
        pooled = tokens.mean(dim=1)
        return self.head(pooled)


# Shape smoke-test only: random tensors verify dimensions; they are NOT MRI data or experiment results.
with torch.no_grad():
    smoke_input = torch.zeros(1, 1, *ROI_SIZE, device=DEVICE)
    cnn_smoke = CNN3DClassifier().to(DEVICE).eval()
    hybrid_smoke = Hybrid3DCNNViT().to(DEVICE).eval()
    print("CNN output shape:", tuple(cnn_smoke(smoke_input).shape))
    print("Hybrid output shape:", tuple(hybrid_smoke(smoke_input).shape))
    del smoke_input, cnn_smoke, hybrid_smoke
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
train_label_ids = train_df["label"].map(LABEL_TO_ID).to_numpy()
class_counts = np.bincount(train_label_ids, minlength=NUM_CLASSES)
if np.any(class_counts == 0):
    raise ValueError(
        "At least one class is absent from the training split. Revisit dataset size and split strategy."
    )
class_weights_np = len(train_label_ids) / (NUM_CLASSES * class_counts)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print("Training class counts:", dict(zip(CLASS_NAMES, class_counts.tolist())))
print("Loss weights:", class_weights.detach().cpu().numpy())

USE_AMP = DEVICE.type == "cuda"


def run_train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    n_samples = 0
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, dtype=torch.long, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        batch_size = labels.shape[0]
        total_loss += float(loss.detach().item()) * batch_size
        n_samples += batch_size
    return total_loss / max(n_samples, 1)


@torch.no_grad()
def evaluate_model(model, loader, calculate_loss=True):
    model.eval()
    all_labels, all_predictions, all_probabilities = [], [], []
    total_loss, n_samples = 0.0, 0
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, dtype=torch.long, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels) if calculate_loss else None
        probabilities = torch.softmax(logits.float(), dim=1)
        predictions = probabilities.argmax(dim=1)
        all_labels.extend(labels.cpu().numpy().tolist())
        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_probabilities.extend(probabilities.cpu().numpy().tolist())
        if loss is not None:
            total_loss += float(loss.item()) * labels.shape[0]
        n_samples += labels.shape[0]

    y_true = np.asarray(all_labels)
    y_pred = np.asarray(all_predictions)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "n": int(len(y_true)),
        "y_true": y_true,
        "y_pred": y_pred,
        "probabilities": np.asarray(all_probabilities),
    }
    if calculate_loss:
        metrics["loss"] = total_loss / max(n_samples, 1)
    return metrics

In [ ]:
def train_model(model, model_name):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    checkpoint_path = OUTPUT_DIR / f"{model_name}_best.pt"
    history = []
    best_val_f1 = -1.0
    epochs_without_improvement = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        train_loss = run_train_epoch(model, train_loader, optimizer, scaler)
        val_metrics = evaluate_model(model, val_loader)
        scheduler.step(val_metrics["macro_f1"])
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_seconds": time.time() - started,
        }
        history.append(row)
        print(
            f"{model_name} | epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train loss {train_loss:.4f} | val loss {val_metrics['loss']:.4f} | "
            f"val macro-F1 {val_metrics['macro_f1']:.4f} | "
            f"val balanced acc {val_metrics['balanced_accuracy']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_f1:
            best_val_f1 = val_metrics["macro_f1"]
            epochs_without_improvement = 0
            torch.save(
                {
                    "model_name": model_name,
                    "model_state_dict": model.state_dict(),
                    "best_val_macro_f1": best_val_f1,
                    "class_names": CLASS_NAMES,
                    "roi_size": ROI_SIZE,
                    "target_spacing": TARGET_SPACING,
                    "seed": SEED,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping {model_name} at epoch {epoch}.")
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(OUTPUT_DIR / f"{model_name}_history.csv", index=False)
    return model, history_df, checkpoint_path


# Run both experiments on the real dataset. If no data/GPU is configured, do not report results.
experiments = {
    "cnn3d_baseline": CNN3DClassifier(),
    "hybrid_cnn_vit": Hybrid3DCNNViT(),
}
trained_models = {}
histories = {}
checkpoints = {}

for experiment_name, model in experiments.items():
    trained_model, history_df, checkpoint_path = train_model(model, experiment_name)
    trained_models[experiment_name] = trained_model
    histories[experiment_name] = history_df
    checkpoints[experiment_name] = checkpoint_path

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, hist in histories.items():
    axes[0].plot(hist["epoch"], hist["train_loss"], label=f"{name} train")
    axes[0].plot(hist["epoch"], hist["val_loss"], linestyle="--", label=f"{name} val")
    axes[1].plot(hist["epoch"], hist["val_macro_f1"], marker="o", label=name)
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[1].set_title("Validation macro-F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro-F1")
for ax in axes:
    ax.grid(True, alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
test_rows = []
test_outputs = {}

for name, model in trained_models.items():
    checkpoint = torch.load(checkpoints[name], map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    metrics = evaluate_model(model, test_loader, calculate_loss=True)
    test_outputs[name] = metrics
    test_rows.append({
        "model": name,
        "test_loss": metrics["loss"],
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
        "test_samples": metrics["n"],
    })

    print(f"\n{name} — classification report")
    print(classification_report(
        metrics["y_true"], metrics["y_pred"],
        labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES, zero_division=0
    ))

    cm = confusion_matrix(
        metrics["y_true"], metrics["y_pred"], labels=list(range(NUM_CLASSES))
    )
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(f"Confusion matrix — {name}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(range(NUM_CLASSES), CLASS_NAMES)
    ax.set_yticks(range(NUM_CLASSES), CLASS_NAMES)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

results_df = pd.DataFrame(test_rows).sort_values("macro_f1", ascending=False)
results_path = OUTPUT_DIR / "test_metrics.csv"
results_df.to_csv(results_path, index=False)
print("Held-out test metrics:")
display(results_df)
print(f"Saved metrics to: {results_path.resolve()}")

In [ ]:
import platform
import monai
import nibabel

run_record = {
    "status": "completed only if all training and test cells finished successfully",
    "seed": SEED,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch_version": torch.__version__,
    "monai_version": monai.__version__,
    "nibabel_version": nibabel.__version__,
    "python_version": platform.python_version(),
    "target_spacing_mm": TARGET_SPACING,
    "roi_size": ROI_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_max": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_scans": len(train_df),
    "validation_scans": len(val_df),
    "test_scans": len(test_df),
    "class_names": CLASS_NAMES,
    "results_csv": str(results_path.resolve()),
}
record_path = OUTPUT_DIR / "experiment_record.json"
with open(record_path, "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2)
print(f"Saved experiment record to: {record_path.resolve()}")
print("Only metrics from a completed run on real held-out scans should be reported as experimental results.")